# Honey Bee Video Motion-Regime Experiments

**This notebook requires the base-distribution files to be synced to your local directory in order to process working data.** Please see `hive_video/README.md` for information on getting started with data.

This notebook documents exploratory video-annotation experiments for the Smith honey bee hive video. The goal is not to identify individual bees perfectly. The goal is to make local motion regimes visible for human review, especially regimes that appear, disappear, or spatially organize around comb/festoon areas.

The source video is:

`data/raw/start04__20190609_175013_side0_top.mp4`

Important context:

- The MP4 is encoded at 25 fps.
- The source appears time-compressed relative to real time; the working hypothesis is that 3 encoded frames correspond to 1 real second.
- The video is assembled out of shuffled chunks, so long-history analyses should reset at segment boundaries unless a join has been accepted.
- Regime overlays are exploratory labels over local image regions, not confirmed behavior labels and not bee identities.


## Literature Grounding

The annotation approach is intended to stay close to papers already collected in the `honey-bee-project` Zotero subcollection.

- Delaplane (2017), *Emergent Properties in the Honey Bee Superorganism*: biological target, especially comb-construction initiation and cell construction.
- Smith et al. (2021/2022), *The dominant axes of lifetime behavioral variation in honey bees* / *Behavioral variation across the days and lives of honey bees*: honey bee movement/substrate axes and the caution that behavior may be continuous rather than sharply classed.
- Wild et al. (2021), *Social networks predict the life and death of honey bees*: low-dimensional derived social/behavioral coordinates and future behavior/task prediction.
- Luxem et al. (2022), *Identifying behavioral structure from deep variational embeddings of animal motion*: label-free temporal embeddings and unsupervised behavioral motifs.
- McKenzie-Smith et al. (2025), *Capturing continuous, long timescale behavioral changes in Drosophila melanogaster postural data*: long-timescale insect behavior and behavioral composition.
- Blanc et al. (2025), *Statistical signature of subtle behavioral changes in large-scale assays*: behavioral distributions in learned latent spaces.
- Uehara et al. (2026) and Fazzari et al. (2024): recent insect video-analysis precedents.

The immediate computational approach is deliberately lighter than VAME: optical flow, boids-like local summary features, PCA, and GMM/KMeans. VAME remains an important conceptual and possible future-method reference if these overlays prove scientifically useful.


## Shared Inputs and Conventions

All experiments below use the raw video in this uv project:

- Input video: `data/raw/start04__20190609_175013_side0_top.mp4`
- Main annotation script: `src/analyze/annotate_motion_regimes.py`
- Output root: `data/qc/`

The annotation script creates:

- `motion_regime_features.csv`: one row per grid cell per time window, including regime probabilities.
- `motion_regime_overlay.mp4`: visual overlay for human review.
- `metadata.json`: exact run settings.

Common interpretation:

- Color: unsupervised local motion-regime cluster.
- Arrow direction: mean local optical-flow vector.
- Arrow length: local motion magnitude.
- Color opacity/grid overlay: visual aid for the inferred local regime, not a behavior truth label.


In [ ]:
from pathlib import Path

VIDEO = Path("start04__20190609_175013_side0_top.mp4")
QC = Path("../qc")
SRC = Path("../src")

print(VIDEO.resolve())
print(QC.resolve())


## Experiment 1: Five-Second Local Motion Regimes

<img src="../data/qc/exp1_reseq_2min_v0p1/exp1.png" width="400" />

### Question

Can local optical-flow features over a five-second history window produce visually meaningful regime annotations over the hive surface?

This experiment was the first pass before adding explicit rotational/angular-neighbor feature weighting.

### Inputs

- Input video: `data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4`
- Preset: `exp1_reseq_2min_v0p1` from `src/analyze/run_analysis.py`
- Instrumentation: `src/analyze/annotate_motion_regimes.py`, `ANALYSIS_VERSION = 0.1.0`
- Feature set: `exp1`, the pre-angular-neighbor baseline feature set. The output CSV still records all computed raw features, but clustering ignores the later angular-neighbor additions.
- Frame range: `start_frame=0`, `duration_frames=3000` (2 encoded minutes at 25 fps)
- Window length: `125` encoded frames, which is 5 encoded seconds at 25 fps.
- Stride: `25` encoded frames, which is 1 encoded second.
- Grid: `16x16`.
- Clusters: `6`.
- PCA: disabled with `pca_components=0`; this baseline clusters standardized `exp1` features directly.
- Optical-flow scale width: `412` pixels.
- Existing resequencing captions are covered with `top_mask_height=72` before overlay text is drawn.

### Methods

1. Decode and downsample frames to a fixed width.
2. Compute dense optical flow between consecutive frames using Farneback optical flow.
3. Split each frame into a spatial grid.
4. For each grid cell and history window, compute local boids-like features:
   - `x_center`, `y_center`
   - `mean_vx`, `mean_vy`
   - `mean_speed`
   - `mean_speed_sq`
   - `std_speed`
   - `active_fraction`
   - `alignment`
   - `divergence`, `curl`
   - `neighbor_speed_contrast`, `neighbor_alignment_contrast`
5. Standardize features.
6. Cluster standardized local cell-window feature vectors directly with a diagonal-covariance GMM.
7. Render an overlay video with colored grid cells and arrows.

### Interpretation Notes

The overlay is a baseline view of local motion regimes. It captures large local motion patches and visually groups some regions, though separation is coarse. This is useful as a comparison point for later experiments that add angular-neighbor features, velocity compression, or longer reviewed frame ranges.


## Experiment 2: Angular/Neighbor-Synchrony Motion Regimes

### Question

Can local angular features separate visually distinct coordinated motion regimes, especially the upper-right comb-area behavior where neighboring bees appear to oscillate within a shared restricted range of directions?

The motivating observation was that some local bee groups are not merely fast/slow or aligned/non-aligned. They appear to share a small subset of the radian circle over a short temporal window, while nearby regions do not.

### Inputs

- Raw video: `start04__20190609_175013_side0_top.mp4`
- Target frame neighborhood from review: around frame `15762`.
- Suggested frame range: `start_frame=14500`, `duration_frames=3000`.
- Window length: `125` frames, preserving the five-second history from Experiment 1.
- Stride: `25` frames.
- Grid: `32x32`, quartering each cell's area relative to a 16x16 grid.
- Clusters: `8`, allowing more local regime categories than Experiment 1.
- PCA components: `8`.
- Optical-flow scale width: `412` for the first pass; `824` is the next resolution knob if needed.

### Methods

Experiment 2 keeps all Experiment 1 features and adds angular/neighbor-synchrony features:

- `direction_concentration`: concentration of recent cell motion directions. High values mean the cell's recent flow directions occupy a narrower part of the direction circle.
- `angular_sweep_std`: variability in frame-to-frame angular change over the window.
- `angular_sweep_abs_mean`: average absolute angular sweep through the window.
- `neighbor_angular_sweep_abs_diff`: mean absolute difference between a cell's angular sweep and adjacent cells' angular sweeps.
- `neighbor_direction_concentration_diff`: contrast between a cell's direction concentration and the mean of adjacent cells.

These are cell-level approximations of the reviewer idea: if nearby bees/patches share a local rotational or oscillatory regime, their angular sweeps should be more similar than random neighboring patches.

Passes at resolutions of 32x32 and 64x64 are presented for comparison.

### Outputs

- `../qc/motion_regimes_5s_32x32_frame15762/motion_regime_features.csv`
- `../qc/motion_regimes_5s_32x32_frame15762/motion_regime_overlay.mp4`
- `../qc/motion_regimes_5s_32x32_frame15762/metadata.json`

### Review Criteria

Human reviewers should compare this overlay against Experiment 1 and ask:

- Does the upper-right comb-area motion separate from lower-half background motion?
- Do colored regimes persist through visually coherent local waves/oscillations?
- Are the added angular features revealing coordinated behavior or just amplifying optical-flow artifacts?
- Does the 32x32 grid improve resolution without making the overlay too noisy?

### Follow-Up Knobs

- Increase optical-flow scale: `--flow-scale-width 824`.
- Compare history windows: `50`, `125`, `250`, and `750` frames.
- Compare cluster counts: `6`, `8`, `10`, `12`.
- Compare `gmm` probabilities with `kmeans` hard clusters.
- Reset history at segment boundaries unless the chunk join has been accepted by review.


In [ ]:
%%bash
# Experiment 2 focused run around the reviewed comb-area frame.
uv run python ../src/annotate_motion_regimes.py \
  start04__20190609_175013_side0_top.mp4 \
  --out ../qc/motion_regimes_5s_32x32_frame15762 \
  --start-frame 14500 \
  --duration-frames 3000 \
  --window-frames 125 \
  --stride-frames 25 \
  --grid-rows 32 \
  --grid-cols 32 \
  --clusters 8 \
  --method gmm \
  --pca-components 8 \
  --flow-scale-width 412


## Experiment 3: Angular-Weighted Long Chunk Run

### Question

Can we make the long `w824` motion-regime overlay more sensitive to coordinated group motion, especially the comb-area waviness seen in `good.png`, without losing the useful regime separation from Experiment 2?

The working hypothesis is that the previous long run already captures much of the visible regime structure, but underweights angular/group-motion features relative to speed, position, and activity. Experiment 3 keeps the long-run settings that worked well and increases the influence of angular and neighbor-synchrony features during clustering.

### Inputs

- Raw video: `start04__20190609_175013_side0_top.mp4`
- Baseline comparison artifacts:
  - `../qc/motion_regimes_long_5s_32x32_w824/good.png`
  - `../qc/motion_regimes_long_5s_32x32_w824/less_good.png`
  - `../qc/motion_regimes_long_5s_32x32_w824/chunks_manifest.csv`
- Frame range: `start_frame=14500`, `duration_frames=180000`.
- Chunking: `chunk_frames=9000`, giving 20 resumable chunks.
- Window length: `125` frames.
- Stride: `25` frames.
- Grid: `32x32`.
- Optical-flow scale width: `824`.
- Clustering: GMM, 8 clusters, diagonal covariance, `reg_covar=1e-4`.

### Methods

Experiment 3 uses the same local optical-flow feature set as Experiment 2, but adds explicit feature weighting before standardization/PCA:

- `--angular-feature-weight`: upweights direction concentration, angular sweep, curl, and angular-neighbor features.
- `--neighbor-feature-weight`: upweights local contrast features relative to adjacent cells.

The first proposed run uses:

- `--angular-feature-weight 2.0`
- `--neighbor-feature-weight 1.5`

This is intentionally conservative. It should make angular/group motion more visible without letting noisy local derivatives dominate the clustering.

### Outputs

Expected output root:

`../qc/motion_regimes_long_5s_32x32_w824_exp3_ang2_neighbor1p5/`

Expected files:

- `chunks_manifest.csv`
- `chunk_*/motion_regime_features.csv`
- `chunk_*/motion_regime_overlay.mp4`
- `motion_regime_overlay_all_chunks.mp4` if `--concat-video` succeeds.

### Review Criteria

Compare Experiment 3 against the previous long `w824` run:

- Does `good.png`-like suspected festoon/comb-area structure become clearer?
- Does the upper-right comb-area waviness separate from lower-half motion more consistently?
- Does `less_good.png` remain appropriately less festoon-like, or does the weighting hallucinate structure?
- Are color regimes stable enough across chunks for human review, while remembering that each chunk currently fits its own GMM?

### Cautions

- This is still a per-chunk clustering run, so colors are not guaranteed globally consistent across the full video.
- Angular derivative features can amplify optical-flow artifacts, especially near edges, glare, or occlusion.
- The safeword file `.safeword` can stop the run cleanly between chunks if it contains `sea cucumber` or `seacucubmer`.


In [ ]:
%%bash
# Experiment 3 long chunked run: w824 with increased angular/group-motion sensitivity.
uv run python ../src/run_motion_regime_chunks.py \
  start04__20190609_175013_side0_top.mp4 \
  --out ../qc/motion_regimes_long_5s_32x32_w824_exp3_ang2_neighbor1p5 \
  --start-frame 14500 \
  --duration-frames 180000 \
  --chunk-frames 9000 \
  --window-frames 125 \
  --stride-frames 25 \
  --grid-rows 32 \
  --grid-cols 32 \
  --clusters 8 \
  --method gmm \
  --gmm-covariance-type diag \
  --gmm-reg-covar 1e-4 \
  --pca-components 8 \
  --flow-scale-width 824 \
  --angular-feature-weight 2.0 \
  --neighbor-feature-weight 1.5 \
  --concat-video


## Experiment 4: Velocity-Compressed High-Resolution Group-Motion Regimes

### Question

Do a few high-velocity bees in sparse/empty regions swamp speed-related features enough to collapse distinctions between upper and lower hive regions? If so, can a nonlinear velocity transform preserve local motion-regime differences better than raw velocity features when we run a higher-resolution pass across the full resequenced video?

### Inputs

This experiment is now a long, restartable full-video run on the resequenced artifact.

- Input video: `data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4`
- Preset: `exp4_reseq_full_highres_v0p1` from `src/analyze/run_analysis.py`
- Instrumentation: `src/analyze/annotate_motion_regimes.py`, `ANALYSIS_VERSION = 0.1.0`
- Frame range: `start_frame=0`, `duration_frames=263474` (the full resequenced video)
- Chunking: `chunk_frames=9000`, giving 30 restartable chunks for the full video.
- Window length: `125` frames, preserving the five-second history window.
- Stride: `25` frames.
- Grid: `48x48`, increasing spatial resolution relative to the earlier `32x32` long runs without jumping all the way to `64x64` cost.
- Optical-flow scale width: `824`.
- GMM with 10 clusters, diagonal covariance, and `reg_covar=1e-4`.
- PCA components: `10`.
- Feature set: `full`.
- Angular/group-motion sensitivity: `--angular-feature-weight 2.0`, `--neighbor-feature-weight 1.5`.
- Existing resequencing captions are covered with `top_mask_height=72` before overlay text is drawn.

### Methods

Experiment 4 adds a configurable velocity transform before scaling/PCA/clustering:

- `--velocity-transform raw`: uncompressed velocity and speed features.
- `--velocity-transform log1p`: compresses large positive speed values and signed velocity magnitudes with signed `log1p`.
- `--velocity-transform sqrt`: milder compression of large velocities.
- `--velocity-transform asinh`: log-like compression that is naturally signed.

This run uses `log1p`, because it directly targets the concern that a few very fast bees may dominate raw speed and velocity features. The raw feature CSV still records the original feature values; the transform affects the clustering feature matrix.

### Outputs

Expected output root:

`data/qc/exp4_reseq_full_highres_v0p1/`

Expected files:

- `analysis_run.json`
- `metadata.json`
- `chunks_manifest.csv`
- `chunk_*/motion_regime_features.csv`
- `chunk_*/motion_regime_overlay.mp4`
- `motion_regime_overlay_all_chunks.mp4` if concatenation succeeds.

### Review Criteria

Compare against Experiments 1, 2, and 3:

- Are upper/lower distinctions less collapsed?
- Are sparse high-speed regions less visually dominant?
- Does suspected comb/festoon structure remain visible?
- Does log compression create artificial regimes in low-motion regions?
- Does the `48x48` grid reveal useful local structure without producing an unreadably noisy overlay?

### Runtime Notes

This is intentionally a long laptop run. It is safe to leave running because the chunk runner writes a manifest after each completed chunk and checks `.safeword` between chunks. To stop cleanly, write `sea cucumber` into `.safeword`; remove the file and rerun the same command to resume.


In [ ]:
%%bash
# Experiment 4 full resequenced-video run: high-resolution log-compressed group motion.
uv run python src/analyze/run_analysis.py \
  exp4_reseq_full_highres_v0p1 \
  --video data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4 \
  --out data/qc/exp4_reseq_full_highres_v0p1


## Experiment Log Template

Use this template for Experiment 3 and later.

### Experiment N: Short Name

### Question

What scientific or review question does this run test?

### Inputs

- Video / segment / frame range:
- Window length and stride:
- Grid resolution:
- Feature families included:
- Clustering method and parameters:

### Methods

What changed relative to earlier experiments?

### Outputs

- Feature CSV:
- Overlay video:
- Metadata:
- Any review notes:

### Reviewer Notes

What looked meaningful, suspicious, noisy, or biologically interesting?
